# EXAI-ResumeIntel: Full XAI Pipeline Walkthrough

**Author:** Mithin Sagar S ([@mithinsagar](https://github.com/mithinsagar))  
**Institution:** Vellore Institute of Technology (VIT)

End-to-end demonstration of the six-component XAI layer:
1. Shapley values (exact for |F| <= 12)
2. LIME local explanation
3. Counterfactual explanations
4. Attention heatmap
5. Feature interactions
6. Natural language insights

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import json

from core.ontology import get_engine
from core.embeddings import SemanticEmbeddingEngine
from core.scorer import ScoringEngine

ontology = get_engine()

try:
    embeddings = SemanticEmbeddingEngine.load('../models/embedding_engine.pkl')
    print('[OK] Embedding engine loaded')
except FileNotFoundError:
    print('[WARN] No trained embeddings found; semantic score will default to 0.5.')
    print('       Run `python -m training.train` first.')
    embeddings = None

scorer = ScoringEngine(ontology, embeddings)

## 1. Score a representative ML resume

In [ ]:
ML_RESUME = '''
Machine Learning Engineer with 5 years experience.

Technical Skills:
- Computer Vision: YOLOv8, COCO dataset, OpenCV, ResNet, Detectron2
- Deep Learning: PyTorch, TensorFlow, ONNX, TensorRT, CUDA
- NLP: BERT, HuggingFace, LangChain, RAG pipelines, FAISS
- MLOps: MLflow, Docker, Kubernetes, AWS SageMaker, TorchServe
- Python: pandas, numpy, scikit-learn, FastAPI

Experience:
Senior ML Engineer at TechCorp (2022-Present)
Built YOLOv8 pipeline achieving 78% mAP on COCO.
Reduced inference latency 65% with TensorRT.

Education: B.Tech Computer Science, VIT University
'''

result = scorer.score(ML_RESUME, 'MACHINE LEARNING ENGINEER')
print(f'Overall Score: {result.overall_score * 100:.1f}%')
print(f'  Ontology:  {result.ontology_score * 100:.1f}%')
print(f'  Semantic:  {result.semantic_score * 100:.1f}%')
print(f'  Depth:     {result.depth_score * 100:.1f}%')
print(f'  Corpus:    {result.corpus_score * 100:.1f}%')

## 2. Shapley Values

In [ ]:
print(f'{"Skill":<32} {"Shapley":>10} {"Impact":<20}')
print('-' * 65)
for s in result.xai_bundle['shap_summary'][:8]:
    print(f'{s["display_name"]:<32} {s["shapley_value"] * 100:>+9.2f}%  {s["impact_label"]:<20}')

## 3. LIME Local Explanation

In [ ]:
print(f'{"Skill":<32} {"LIME weight":>12} {"Direction":>10}')
print('-' * 60)
for r in result.xai_bundle['lime_results'][:8]:
    print(f'{r["display_name"]:<32} {r["lime_weight"]:>+12.4f}  {r["direction"]:>10}')

## 4. Counterfactual Scenarios

In [ ]:
for cf in result.xai_bundle['counterfactuals']:
    print(f'\n{cf["description"]}')
    print(f'  Score:    {cf["score_before"]}% -> {cf["score_after"]}% (+{cf["gain"]:.1f}%)')
    print(f'  Priority: {cf["priority"]}')

## 5. Feature Interactions

In [ ]:
for i in result.xai_bundle['feature_interactions']:
    print(f'{i["label"]:<50} {i["interaction"]:>+.4f}  [{i["synergy"]}]')

## 6. Natural Language Summary

In [ ]:
print(result.xai_bundle['nl_overall'])
print('\n--- SHAP insights ---')
for insight in result.xai_bundle['nl_shap_insights']:
    print(f'* {insight}')
print('\n--- LIME insight ---')
print(result.xai_bundle['nl_lime_insight'])
print('\n--- Counterfactual insights ---')
for insight in result.xai_bundle['nl_counterfactual_insights']:
    print(f'* {insight}')

---
**End of walkthrough.** For the complete research pipeline with all figures, see `EXAI_Resume_Intelligence_FINAL.ipynb`.